In [ ]:
# pip install requests json python-binance pandas openpyxl tqdm
# python = 3.12

In [1]:
# 관련 docs
# [01]. binance api
#       https://developers.binance.com/docs/binance-spot-api-docs

In [1]:
import requests
import json
from binance.client import Client

import pandas as pd
import numpy as np
import openpyxl

import os
import time

from tqdm import tqdm

In [2]:
# 주의
KEYS = pd.read_excel("private_keys.xlsx")
api_key = KEYS['API Key'][0]
api_secret = KEYS['Secret Key'][0]

client = Client(api_key, api_secret)


In [12]:
KLINE_COLUMNS = [
    'Open time', 'Open', 'High', 'Low', 'Close', 'Volume', 
    'Close time', 'Quote asset volume', 'Number of trades', 
    'Taker buy base asset volume', 'Taker buy quote asset volume', 'Ignore'
]

class BinanceDataCollector:
    """
    바이낸스 API를 사용하여 심볼 정보 및 캔들스틱 데이터를 수집하는 클래스입니다.
    """
    def __init__(self, api_key, api_secret):
        # binance.client.Client 객체를 인스턴스 변수로 저장
        self.client = Client(api_key, api_secret)
        self.all_symbols = self.get_all_symbols()

    def get_all_symbols(self):
        """
        Binance 상장된 모든 Symbol 추출목적
        """
        info = self.client.get_exchange_info() # 심볼 + 필터
        
        symbols = [x['symbol'] for x in info['symbols']]
        print(f"전체 심볼 개수 : {len(symbols)}")
        return symbols
    def is_valid_symbol(self, symbol, interval, start_str, end_str):
        """
        시작, 중간, 끝 지점에 데이터가 존재하는지 체크 (데이터 누수 및 상장 기간 확인)
        """
        try:
            # 1. 날짜 계산
            start_ts = pd.to_datetime(start_str)
            end_ts = pd.to_datetime(end_str) if end_str else pd.Timestamp.now()
            mid_ts = start_ts + (end_ts - start_ts) / 2

            # 점검할 타임스탬프 리스트 (ms 단위 변환)
            check_points = [start_ts, mid_ts, end_ts]
            
            for ts in check_points:
                # 해당 시점에 캔들이 딱 1개라도 있는지 확인 (limit=1)
                check = self.client.get_klines(
                    symbol=symbol,
                    interval=interval,
                    startTime=int(ts.timestamp() * 1000),
                    limit=1
                )
                
                if not check:
                    # 데이터가 한 지점이라도 없으면 탈락
                    return False
            
            return True
        except Exception as e:
            print(f"점검 중 오류 ({symbol}): {e}")
            return False
        
    def get_binance_klines(self, symbol, interval, start_str, end_str=None):
        # start_str과 end_str은 날짜 문자열 (예: '1 Jan, 2024').

        """
        Binance API에서 캔들스틱 데이터 추출
        , API 과다호출로 인한 ban을 받기 위해서 1회 호출시 symbol개수 한정
        """
        try:
            klines = self.client.get_historical_klines(
                symbol, 
                interval, 
                start_str, 
                end_str,
                limit = None
            )
            if not klines:
                print(f"{symbol} : 심볼 데이터 없음. 이유는 체크필요.")
                return None
            
            # DataFrame으로 변환하고 컬럼 이름 명시적으로 설정
            data = pd.DataFrame(klines, columns=KLINE_COLUMNS)

            # 데이터 타입 변환 및 인덱스 설정 (기존 get_binance_klines 로직 통합)
            data['Open time'] = pd.to_datetime(data['Open time'], unit='ms')
            data['Close time'] = pd.to_datetime(data['Close time'], unit='ms')
            
            numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
            for col in numeric_cols:
                # 숫자형 컬럼을 float으로 변환
                data[col] = pd.to_numeric(data[col])
                
            data = data.set_index('Open time')
            return data
        
        except Exception as e:
            print(f"{symbol} : 데이터 추출 오류 코드({e})")
            return None
        finally:
            sleep_time = np.random.randint(1,100+1)/100
            time.sleep(sleep_time) # ban 방지
            
    def get_binance_klines_1hr(self, symbol, interval, start_str, end_str=None):
        """
        Binance API에서 1,000개 제한을 넘어서 전체 기간 데이터를 추출하는 로직
        """
        try:
            # 1. 시작 및 종료 지점을 밀리초(ms) 단위 타임스탬프로 변환
            start_ts = int(pd.to_datetime(start_str).timestamp() * 1000)
            end_ts = int(pd.to_datetime(end_str).timestamp() * 1000) if end_str else int(time.time() * 1000)
            
            all_klines = []
            
            while True:
                # 최대 1,000개씩 요청
                klines = self.client.get_klines(
                    symbol=symbol,
                    interval=interval,
                    startTime=start_ts,
                    endTime=end_ts,
                    limit=1000
                )

                if not klines:
                    break
                
                all_klines.extend(klines)
                
                # 마지막으로 가져온 캔들의 'Close time'을 확인
                last_close_time = klines[-1][6]
                
                # 다음 요청의 시작 지점은 마지막 캔들의 종료 시간 + 1ms
                start_ts = last_close_time + 1
                
                # 만약 가져온 데이터가 1,000개 미만이면 더 이상 데이터가 없는 것
                if len(klines) < 1000 or start_ts >= end_ts:
                    break
                
                # API 가부하 방지를 위한 미세 지연 (루프 내부)
                time.sleep(0.05)

            if not all_klines:
                print(f"{symbol} : 해당 기간에 데이터가 없습니다.")
                return None
            
            # DataFrame 변환
            data = pd.DataFrame(all_klines, columns=KLINE_COLUMNS)

            # 전처리
            data['Open time'] = pd.to_datetime(data['Open time'], unit='ms')
            data['Close time'] = pd.to_datetime(data['Close time'], unit='ms')
            
            numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
            for col in numeric_cols:
                data[col] = pd.to_numeric(data[col])
                
            data = data.set_index('Open time')
            return data
        
        except Exception as e:
            print(f"{symbol} : 데이터 추출 중 오류 발생 - {e}")
            return None
        

In [13]:
data_collector = BinanceDataCollector(api_key, api_secret)

# INTERVAL = Client.KLINE_INTERVAL_1HOUR
INTERVAL = Client.KLINE_INTERVAL_1HOUR

START_DATE = "1 March, 2024"
END_DATE = "12 April, 2026"

전체 심볼 개수 : 3559


In [5]:
# 데이터 dict형식으로 받을 예정
all_klines_data = {}

print("==========Data Extract Process is started.==========")

df_list =[]

# 인스턴스 변수에 저장된 전체 심볼 목록 사용
all_symbols = data_collector.all_symbols
all_symbols = [x for x in all_symbols]
all_symbols = [x for x in all_symbols if x.endswith('USDT')]
print(len(all_symbols)) # 658 - 260410

clean_symbols = []

print("========== Step 1: 기간 검증 필터링 시작 ==========")
for symbol in tqdm(all_symbols, desc="유효성 검사"):
    # 3점 테스트 통과 여부 확인
    if data_collector.is_valid_symbol(symbol, INTERVAL, START_DATE, END_DATE):
        clean_symbols.append(symbol)
    
    # API 과부하 방지를 위한 미세 지연
    time.sleep(0.01)

print(f"검증 통과 심볼: {len(clean_symbols)} / {len(all_symbols)}")

==========Data Extract Process is started.==========
658
========== Step 1: 기간 검증 필터링 시작 ==========


유효성 검사: 100%|██████████| 658/658 [01:40<00:00,  6.53it/s]

검증 통과 심볼: 440 / 658


In [ ]:
print("========== Step 2: 실제 데이터 추출 시작 ==========")
df_list = []
i = 0
for symbol in tqdm(clean_symbols, desc="데이터 수집"):
    df_klines = data_collector.get_binance_klines_1hr(symbol, INTERVAL, START_DATE, END_DATE)
    
    if df_klines is not None:
        df_klines['Symbol'] = symbol
        df_list.append(df_klines)
    time.sleep(0.01) # API 과부하 방지 위한 미세 지연
    i += 1
    if i % 10 == 0:
        print(f"{symbol} : ({len(df_klines)}개) 데이터 추출 완료")
        time.sleep(3+np.random.uniform(0, 2)) # 10 symbols마다 5초 휴식
df_all = pd.DataFrame() # 364개
for i in range(len(df_list)):
    df_all = pd.concat([df_all, df_list[i]], axis=0, ignore_index=True)
# df_all.to_excel("./data/binance_klines_1hr_260412_1.xlsx", index=False)
df_all.to_parquet("./data/binance_klines_1hr_260412_1.parquet", index=False)


========== Step 2: 실제 데이터 추출 시작 ==========


데이터 수집:   2%|▏         | 9/440 [00:21<17:03,  2.37s/it]

IOTAUSDT : (18529개) 데이터 추출 완료


데이터 수집:   4%|▍         | 19/440 [00:49<17:05,  2.43s/it]

HOTUSDT : (18529개) 데이터 추출 완료


...

SENTUSDT : (1908개) 데이터 추출 완료


데이터 수집: 100%|█████████▉| 438/440 [17:05<00:00,  3.82it/s]

USDSUSDT : (65개) 데이터 추출 완료


데이터 수집: 100%|██████████| 440/440 [17:09<00:00,  2.34s/it]


ValueError: This sheet is too large! Your sheet size is: 6495173, 12 Max sheet size is: 1048576, 16384

In [15]:
df_all.to_parquet("./data/binance_klines_1hr_260412_1.parquet", index=False)


In [20]:
df_len = pd.DataFrame(df_all['Symbol'].value_counts())
idx_all = df_len.loc[df_len['count']==df_len['count'].max(),:].index
df_all_260412 = df_all.loc[df_all['Symbol'].isin(idx_all),:]

In [24]:
print(sum([1 for _ in  idx_all]))
# 271개 심볼
print(df_all_260412.shape)
# (5021359, 12)

271
(5021359, 12)


In [25]:
df_all_260412.to_parquet("./data/binance_klines_1hr_260412.parquet", index=False)

In [ ]:
print("========== Step 2: 실제 데이터 추출 시작 ==========")
df_list = []
for symbol in tqdm(clean_symbols[:364], desc="데이터 수집"):
    df_klines = data_collector.get_binance_klines(symbol, INTERVAL, START_DATE, END_DATE)
    
    if df_klines is not None:
        df_klines['Symbol'] = symbol
        df_list.append(df_klines)

df_all = pd.DataFrame() # 364개
for i in range(len(df_list)):
    df_all = pd.concat([df_all, df_list[i]], axis=0, ignore_index=True)
df_all.to_excel("./data/binance_klines_260410_1.xlsx", index=False)
df_all.to_parquet("./data/binance_klines_260410_1.parquet", index=False)



========== Step 2: 실제 데이터 추출 시작 ==========


데이터 수집:   0%|          | 0/448 [00:00<?, ?it/s]

BTCUSDT : (701개) 데이터 추출 완료


데이터 수집:   0%|          | 1/448 [00:00<05:24,  1.38it/s]

ETHUSDT : (701개) 데이터 추출 완료


...

KERNELUSDT : (353개) 데이터 추출 완료
WCTUSDT : (352개) 데이터 추출 완료


데이터 수집:  81%|████████▏ | 364/448 [03:55<00:32,  2.58it/s]

HYPERUSDT : (345개) 데이터 추출 완료
INITUSDT : (343개) 데이터 추출 완료


데이터 수집:  81%|████████▏ | 364/448 [03:55<00:54,  1.55it/s]


KeyboardInterrupt: 

In [31]:
print("========== Step 2: 실제 데이터 추출 시작 ==========")
df_list = []
for symbol in tqdm(clean_symbols[364:], desc="데이터 수집"):
    df_klines = data_collector.get_binance_klines(symbol, INTERVAL, START_DATE, END_DATE)
    
    if df_klines is not None:
        df_klines['Symbol'] = symbol
        df_list.append(df_klines)
df_all = pd.DataFrame() # 364개
for i in range(len(df_list)):
    df_all = pd.concat([df_all, df_list[i]], axis=0, ignore_index=True)
df_all.to_excel("./data/binance_klines_260410_2.xlsx", index=False)
df_all.to_parquet("./data/binance_klines_260410_2.parquet", index=False)


========== Step 2: 실제 데이터 추출 시작 ==========


데이터 수집:   0%|          | 0/84 [00:00<?, ?it/s]

INITUSDT : (343개) 데이터 추출 완료


데이터 수집:   2%|▏         | 2/84 [00:01<00:46,  1.77it/s]

SIGNUSDT : (339개) 데이터 추출 완료
STOUSDT : (335개) 데이터 추출 완료


...

데이터 수집:  98%|█████████▊| 82/84 [00:53<00:01,  1.35it/s]

XAUTUSDT : (7개) 데이터 추출 완료


데이터 수집: 100%|██████████| 84/84 [00:54<00:00,  1.66it/s]

USDSUSDT : 심볼 데이터 없음. 이유는 체크필요.


데이터 수집: 100%|██████████| 84/84 [00:54<00:00,  1.54it/s]


In [32]:
df_all_1 = pd.read_parquet("./data/binance_klines_260410_1.parquet")
df_all_2 = pd.read_parquet("./data/binance_klines_260410_2.parquet")
df_all_260410 = pd.concat([df_all_1, df_all_2], axis=0, ignore_index=True)
df_all_260410.to_parquet("./data/binance_klines_1day_260410.parquet", index=False)

In [12]:


# # test
# # for symbol in tqdm(all_symbols,miniters=100, desc="처리 중"):
# for symbol in tqdm(all_symbols,miniters=100, desc="처리 중"):
#     df_klines = data_collector.get_binance_klines(symbol, INTERVAL, START_DATE)
    
#     if df_klines is not None:
#         df_klines['Symbol'] = symbol
#         # all_klines_data[symbol] = df_klines
#         df_list.append(df_klines)

# if df_list:
#     df_final = pd.concat(df_list)
#     df_final = df_final.sort_values(by = ['Symbol', 'Open time'])
#     print("==========Data Extract Process is done.==========")
#     print(f"총 수집된 행(Row) 수: {len(df_final)}")
#     print(df_final.head())
# else :
#     print('no data')

In [41]:
df_all_260410.drop_duplicates(inplace=True)

In [43]:
df_all_260410.shape

(254445, 12)

In [52]:
df_len = pd.DataFrame(df_all_260410['Symbol'].value_counts())
idx_all = df_len.loc[df_len['count']==df_len['count'].max(),:].index
df_all_260410 = df_all_260410.loc[df_all_260410['Symbol'].isin(idx_all),:]
df_all_260410.to_parquet("./data/binance_klines_1day_260410.parquet", index=False)

In [61]:
df_all_260410.head(10).to_clipboard()

In [62]:
df_all_260410

,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,Ignore,Symbol
0,60672.01000,60841.63000,56552.82000,58364.97000,8.116647e+04,2024-05-01 23:59:59.999,4705572361.74020020,2401089,38203.57377000,2213404278.59001090,0,BTCUSDT
1,58364.97000,59625.00000,56911.84000,59060.61000,4.758382e+04,2024-05-02 23:59:59.999,2778840305.05057860,1572898,23240.75370000,1356148775.88673650,0,BTCUSDT
2,59060.60000,63333.00000,58811.32000,62882.01000,4.362840e+04,2024-05-03 23:59:59.999,2658193708.34790420,1558661,22145.44334000,1349687455.59576090,0,BTCUSDT
3,62882.01000,64540.00000,62541.03000,63892.04000,2.436869e+04,2024-05-04 23:59:59.999,1547468697.87657970,1113509,12671.20599000,804768764.30625010,0,BTCUSDT
4,63892.03000,64646.00000,62822.17000,64012.00000,1.852675e+04,2024-05-05 23:59:59.999,1182572239.53061830,992921,8954.93439000,571783428.23947070,0,BTCUSDT
...,...,...,...,...,...,...,...,...,...,...,...,...
203986,0.00332,0.00346,0.00323,0.00323,1.611156e+08,2026-03-28 23:59:59.999,539710.58046400,7309,90934686.50000000,304774.74032400,0,REZUSDT
203987,0.00323,0.00342,0.00322,0.00330,1.455667e+08,2026-03-29 23:59:59.999,484320.57909200,7509,76194307.90000000,253479.26570600,0,REZUSDT
203988,0.00330,0.00340,0.00322,0.00331,1.935584e+08,2026-03-30 23:59:59.999,641264.29829700,9749,104300996.60000000,346006.97146500,0,REZUSDT
203989,0.00331,0.00337,0.00326,0.00331,1.823615e+08,2026-03-31 23:59:59.999,603587.84836600,8401,103575312.40000000,342874.95438100,0,REZUSDT
